# Import data from UCI Machine Learning Repository
### Dataset: Online Retail
https://archive.ics.uci.edu/dataset/352/online+retail

In [63]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [64]:
from ucimlrepo import fetch_ucirepo 
import duckdb
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
df = online_retail.data.original 

In [65]:
df.shape

(541909, 8)

# Data Cleaning
Using SQL to clean data to demonstrate capability

In [ ]:
# Filtering dataset
df = duckdb.sql(
    """
    WITH main_unfiltered AS(
        SELECT
            Description                                                                             AS description,
            Quantity                                                                                AS quantity,
            CAST(STRPTIME(InvoiceDate, '%m/%d/%Y %H:%M') AS DATE)                                   AS invoice_date,
            UnitPrice                                                                               AS unit_price,
            CAST(CustomerID AS STRING)                                                              AS customer_id,
            Country                                                                                 AS country,
            InvoiceNo                                                                               AS invoice_number,
            MAX(CAST(STRPTIME(InvoiceDate, '%m/%d/%Y %H:%M') AS DATE)) OVER ()                      AS global_max_date
        FROM
            df
    ),

    main AS (
        SELECT
            *
        FROM
            main_unfiltered
        WHERE
            invoice_date < DATE_ADD(global_max_date, INTERVAL '-3 MONTH') -- Filter out the last 90 days, this will be used for target variables
            AND description = UPPER(description) -- Filtering to just Products, Products are capitalised and adjustments such as discounts are lowercase
            AND quantity >= 1
            AND unit_price >= 0.01
    ),

    max_invoice_date AS (
        SELECT
            MAX(invoice_date) AS max_date
        FROM 
            main
    ),

    customer_lifespan AS (
        SELECT
            customer_id,
            DATEDIFF('day', MIN(invoice_date), MAX(invoice_date)) AS customer_lifespan_days,
            COUNT(DISTINCT invoice_number) AS life_time_orders,
            SUM(quantity*unit_price) AS total_spend, 
            MAX(invoice_date) AS last_order_date 
        FROM
            main
        WHERE
            customer_id IS NOT NULL
        GROUP BY
            customer_id
    ),

    discount AS (
        SELECT
            customer_id,
            invoice_date,
            SUM(quantity*unit_price) AS discount,
            COUNT(invoice_number) AS number_of_discounts 
        FROM
            main_unfiltered
        WHERE
            description = 'Discount'
            AND invoice_date < DATE_ADD(global_max_date, INTERVAL '-3 MONTH')
        GROUP BY
            customer_id, 
            invoice_date
    ),

    target_variable AS (
        SELECT DISTINCT
            customer_id,
            0 as customer_churned
        FROM 
            main_unfiltered
        WHERE
            invoice_date >= DATE_ADD(global_max_date, INTERVAL '-3 MONTH') -- Filter to the last 90 days of data
    ),
    
    final AS (
        SELECT 
            a.customer_id,
            SUM(a.quantity * a.unit_price)                                  AS total_spend
            MAX(b.customer_lifespan_days)                                   AS customer_age_days,
            MAX(b.life_time_orders)                                         AS total_orders,
            SUM(a.quantity * a.unit_price) / MAX(b.life_time_orders)        AS average_order_value,
            MAX(b.customer_lifespan_days) / (MAX(b.life_time_orders) -1)    AS order_frequency,
            SUM(IFNULL(c.discount * -1, 0))                                 AS total_discount_value,
            MAX(IFNULL(c.number_of_discounts, 0))                           AS orders_with_discounts,
            DATEDIFF('day', MAX(a.invoice_date), MAX(e.max_date))           AS days_since_last_order,
            MAX(IFNULL(d.customer_churned, 1))                              AS customer_churned
        FROM 
            main a
        LEFT JOIN
            customer_lifespan b USING (customer_id)
        LEFT JOIN
            discount c USING (customer_id) -- Can sum create the count of discounts per customer then get the sum for total discount and avg discount
        LEFT JOIN
            target_variable d USING (customer_id)
        CROSS JOIN 
            max_invoice_date e
        GROUP BY ALL
    )

    SELECT * FROM final
    """
).df()

df

,customer_id,customer_age_days,total_orders,total_discount_value,orders_with_discounts,days_since_last_order,customer_churned
0,16672.0,167,17,2610.00,1,7,0
1,16422.0,270,39,32900.46,2,7,0
2,17068.0,274,16,0.00,0,2,0
3,15281.0,197,8,0.00,0,44,0
4,13993.0,230,6,0.00,0,8,0
...,...,...,...,...,...,...,...
3355,13809.0,0,1,0.00,0,213,1
3356,12361.0,0,1,0.00,0,195,1
3357,13216.0,0,1,0.00,0,175,1
3358,12453.0,0,1,0.00,0,42,1
